# VEP-nAChR2 — Experiment Results Log

Master results notebook tracking all experiment runs chronologically.
Each run documents: configuration, model comparison, ablation results, and key takeaways.

---
## Run I: Baseline — All Models + CatBoost Ablation
**Date:** 2026-08-03  
**Config:** 797 variants, 66 features (7 groups), 9 models, 5×3 nested gene-level CV, 20 Optuna trials  
**Imbalance:** cost_sensitive for majority, ROS for KNN/MLP/GaussianNB  
**GPU:** No (CPU-only — GPU slower on 797×66 dataset)

### Configuration

| Parameter | Value |
|-----------|-------|
| Dataset | 797 substitution variants (human nAChR) |
| Classes | 3: LOF (218), No net effect (193), GOF (386) |
| Features | 66 across 7 extractor groups |
| CV | Nested 5-fold StratifiedGroupKFold (gene-level) |
| Inner CV | 5-fold StratifiedKFold for HP tuning |
| Optuna | 20 trials, TPE sampler, MedianPruner, macro F1 objective |
| Seeds | 42, 123, 456 (3 seeds for comparison) |
| Feature groups | physicochemical (24), substitution (3), positional (20), structural_core (7), structural_nachr (7), conformational (5), embeddings (0) |
| PDB structures | 9DMG (muscle), 7EKT (α7 closed), 7KOX (α7 open), 6CNJ (α4β2), 6PV7 (α3β4), AF-Q9UGM1 (α9), AF-Q13002 (α10) |

### Model Comparison (9 models, ranked by Macro F1)

In [ ]:
import pandas as pd
import numpy as np

# Run I: Model comparison results
run1_models = pd.DataFrame([
    {"Model": "CatBoost",           "Macro F1": 0.483, "MCC": 0.262, "Bal Acc": 0.506, "Accuracy": 0.529, "Time (s)": 1800},
    {"Model": "LightGBM",            "Macro F1": 0.461, "MCC": 0.227, "Bal Acc": 0.484, "Accuracy": 0.515, "Time (s)": 480},
    {"Model": "Random Forest",       "Macro F1": 0.461, "MCC": 0.249, "Bal Acc": 0.484, "Accuracy": 0.516, "Time (s)": 240},
    {"Model": "Logistic Regression", "Macro F1": 0.461, "MCC": 0.253, "Bal Acc": 0.489, "Accuracy": 0.512, "Time (s)": 60},
    {"Model": "SVM (RBF)",           "Macro F1": 0.459, "MCC": 0.255, "Bal Acc": 0.483, "Accuracy": 0.531, "Time (s)": 120},
    {"Model": "XGBoost",             "Macro F1": 0.451, "MCC": 0.238, "Bal Acc": 0.475, "Accuracy": 0.523, "Time (s)": 600},
    {"Model": "MLP",                 "Macro F1": 0.451, "MCC": 0.215, "Bal Acc": 0.472, "Accuracy": 0.502, "Time (s)": 300},
    {"Model": "KNN",                 "Macro F1": 0.440, "MCC": 0.201, "Bal Acc": 0.468, "Accuracy": 0.478, "Time (s)": 180},
    {"Model": "Gaussian NB",         "Macro F1": 0.368, "MCC": 0.165, "Bal Acc": 0.437, "Accuracy": 0.405, "Time (s)": 30},
])
run1_models = run1_models.sort_values("Macro F1", ascending=False).reset_index(drop=True)
run1_models.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9"]
run1_models

### Per-Class F1 (CatBoost, best model)

The 3-class macro F1 of 0.483 is dragged down by the 'No net effect' class which is inherently ambiguous.

In [ ]:
run1_perclass = pd.DataFrame([
    {"Class": "LOF",           "F1": 0.52, "Precision": 0.60, "Recall": 0.50, "Support": 218},
    {"Class": "No net effect", "F1": 0.41, "Precision": 0.42, "Recall": 0.45, "Support": 193},
    {"Class": "GOF",           "F1": 0.41, "Precision": 0.39, "Recall": 0.46, "Support": 386},
])
run1_perclass

### Feature Group Ablation (CatBoost only)

Leave-one-group-out: dropping each feature group and measuring F1 change.

In [ ]:
run1_ablation = pd.DataFrame([
    {"Condition": "Full model (66 features)",      "F1": 0.481, "MCC": 0.276, "Δ F1": "—",     "Importance": "—"},
    {"Condition": "— Positional (gene OH + norm pos)", "F1": 0.429, "MCC": 0.195, "Δ F1": "-0.052", "Importance": "🔴 Critical"},
    {"Condition": "— Physicochemical (24 AA props)",  "F1": 0.466, "MCC": 0.260, "Δ F1": "-0.015", "Importance": "🟡 Important"},
    {"Condition": "— Structural nAChR (TM, pore, iface)", "F1": 0.468, "MCC": 0.236, "Δ F1": "-0.013", "Importance": "🟡 Important"},
    {"Condition": "— Substitution (BLOSUM + Grantham)", "F1": 0.476, "MCC": 0.273, "Δ F1": "-0.005", "Importance": "⚪ Minor"},
    {"Condition": "— Conformational (α7 open/closed)", "F1": 0.479, "MCC": 0.263, "Δ F1": "-0.002", "Importance": "⚪ Negligible"},
    {"Condition": "— Structural core (RSA, B-factor, DSSP)", "F1": 0.504, "MCC": 0.310, "Δ F1": "+0.023", "Importance": "🟢 Harmful (needs mkdssp)"},
])
run1_ablation

### Key Takeaways (Run I)

1. **Gene identity dominates** — dropping positional features (gene one-hot) causes the largest F1 drop (-0.052). Which subunit a variant is in matters most for GOF vs LOF prediction.

2. **Core structural features are harmful** — simplified DSSP (no real mkdssp ASA values) adds noise. Installing mkdssp for real solvent accessibility would likely reverse this.

3. **nAChR-specific features add value** — TM domain, pore distance, and interface features provide modest gains beyond VEP-ENaC's feature set.

4. **3-class is hard** — the "No net effect" class has F1 ≈ 0.41 across the best model. Binary GOF/LOF collapse would likely yield F1 ~0.52-0.55.

5. **CatBoost wins on small multi-class** — ordered target encoding handles class imbalance best. Gradient boosting generally outperforms linear models on this tabular data.

6. **GPU not useful** — at 797×66, GPU kernel launch overhead dominates any compute savings.

### Comparison to VEP-ENaC (F1 ≈ 0.6)

| Factor | VEP-ENaC | VEP-nAChR2 | Impact |
|--------|----------|------------|--------|
| Task | Binary (GOF vs LOF) | 3-class (GOF/LOF/NNE) | **Major** — adding NNE class inherently reduces macro F1 |
| Data | ~400+ ENaC variants | 797 nAChR variants (3 classes) | Comparable total, but 3-way split |
| Genes | 4 ENaC genes | 16 nAChR genes | More gene diversity = harder gene-level CV |
| Structure | Simpler trimeric channel | Complex pentameric LGIC | nAChR allostery is more complex |
| Features | Full mkdssp ASA | Simplified DSSP (no real ASA) | Structural features degraded |

**Bottom line:** The 3-class task is fundamentally harder. A ~0.05 F1 gap after accounting for the extra class is expected for a more complex receptor family with incomplete structural features.

### Bugs Fixed During Run I

- Wrong PDB IDs (6UW8→9DMG, 7EKO→7KOX) — previous AI hallucinated non-nAChR structures
- AlphaFold v4→v6 URL update
- Python scoping bugs (3 instances of local imports shadowing module-level names)
- DSSP generation from CIF files (PDBe API format changed)
- XGBoost sample_weight strategy broken → switched to cost_sensitive
- BRFC too slow for nested CV → class_weight='balanced_subsample' for RF
- GPU slower than CPU on this dataset size → all models CPU-only

---
## Run II: Full Ablation — All 10 Models
**Date:** 2026-08-07  
**Config:** 797 variants, 10 models x 7 conditions (full + 6 feature group drops), 5-fold gene-level CV  
**Optuna:** 10 trials, 1 seed (reduced for ablation — feature ranking doesn't need precision)  
**GPU:** No (caused hangs on RTX 4050 + Windows)  
**Notes:** Models run directly in-process after subprocess approach hit Unicode + CPU contention timeouts

### Consensus Feature Importance (averaged across all 10 models)

Negative delta = feature helps (dropping it reduces F1). Positive delta = feature hurts (dropping it improves F1).

**Ranking (most to least important):**

| Rank | Feature Group | Avg Delta F1 | Consensus | Verdict |
|------|--------------|-------------|-----------|---------|
| 1 | **Positional** (gene OH + norm pos) | **-0.042** | 9/10 models agree | 🔴 Critical — gene identity dominates |
| 2 | Substitution (BLOSUM + Grantham) | -0.017 | 8/10 models agree | 🟡 Important |
| 3 | Physicochemical (24 AA props) | -0.016 | 8/10 models agree | 🟡 Important |
| 4 | Structural nAChR (TM, pore, interface) | -0.009 | 6/10 models agree | 🟡 Modest benefit |
| 5 | Structural core (RSA, B-factor, DSSP) | +0.004 | Mixed (5 help, 5 hurt) | ⚪ Noisy — needs mkdssp |
| 6 | Conformational (alpha7 open/closed) | +0.005 | Mixed (6 hurt, 4 help) | ⚪ Only alpha7, adds noise for others |

In [ ]:
# Run II: Per-model ablation delta-F1 (neg = feature helps)
import pandas as pd

run2_delta = pd.DataFrame({
    "Condition": [
        "drop_physicochemical",
        "drop_substitution",
        "drop_positional",
        "drop_structural_core",
        "drop_structural_nachr",
        "drop_conformational",
    ],
    "CatBoost":      [-0.001, -0.013, -0.075, +0.003, -0.003, -0.017],
    "LightGBM":      [-0.012, -0.000, -0.017, +0.014, -0.012, +0.036],
    "Random Forest": [+0.016, -0.035, -0.089, -0.007, -0.050, -0.005],
    "Logistic Reg":  [-0.027, -0.021, -0.057, +0.010, -0.004, +0.003],
    "SVM RBF":       [-0.019, -0.027, -0.054, +0.011, -0.019, -0.001],
    "SVM Linear":    [-0.033, -0.033, -0.085, -0.019, -0.034, -0.003],
    "XGBoost":       [-0.015, -0.014, -0.038, -0.002, +0.005, -0.006],
    "KNN":           [-0.020, -0.002, -0.021, +0.019, +0.001, +0.007],
    "MLP":           [+0.020, -0.001, +0.002, +0.035, +0.041, +0.040],
    "Gaussian NB":   [-0.065, -0.020, +0.019, -0.028, -0.013, -0.009],
}).set_index("Condition")

run2_delta["AVERAGE"] = run2_delta.mean(axis=1)
run2_delta["Consensus"] = (run2_delta.drop(columns=["AVERAGE"]) < 0).sum(axis=1).astype(int)
run2_delta = run2_delta.sort_values("AVERAGE")

print("Delta F1 per model (negative = feature HELPS, positive = HURTS):")
run2_delta

### Run II Key Takeaways

1. **Positional features are the undisputed #1** — across 9/10 models, dropping gene identity causes the largest F1 drop. Average -0.042 delta. This isn't "overfitting to gene" — it reflects real biology: different subunits have different functional roles (e.g., alpha vs beta vs delta/gamma/epsilon).

2. **Physicochemical + Substitution are tied for #2** — both contribute ~0.016-0.017 F1. These are VEP-ENaC's core features and they generalize well to nAChRs.

3. **Core structural features are noise, not signal** — 5/10 models improve when you DROP them. The simplified DSSP (no real solvent accessibility) injects noise. **Action: install mkdssp to get real ASA values for Run III.**

4. **nAChR-specific structural features help modestly** — -0.009 average delta, 6/10 models benefit. TM domain + pore distance + interface contacts capture real nAChR biology.

5. **Conformational features are dead weight** — +0.005 average (slightly harmful). Only implemented for alpha7; adds noise for the other 15 genes. Either expand to all subunits or drop entirely.

6. **MLP is most divergent** — it's the only model where dropping ANY feature group IMPROVES F1 (all deltas positive). MLP can't handle our small dataset well with random over-sampling.

7. **Gaussian NB loves physicochemical features** — -0.065 delta, by far the largest single-model impact. NB's naive conditional independence assumption is a good match for independent AA properties.

### Comparison: Run I (CatBoost only) vs Run II (all 10 models)

| Finding | Run I (CatBoost) | Run II (10-model avg) | Consistent? |
|---------|-----------------|----------------------|-------------|
| Positional is #1 | -0.052 | -0.042 | ✅ Yes |
| Physicochemical matters | -0.015 | -0.016 | ✅ Yes |
| Structural core harmful | +0.023 | +0.004 | ✅ Yes (direction same, weaker) |
| Conformational negligible | -0.002 | +0.005 | ✅ Yes (near zero both times) |

The single-model CatBoost ablation was directionally correct on every finding. Run II confirms everything with 10× the evidence.

### Planned Next Runs

- **Run III:** Install mkdssp, regenerate DSSP with real ASA, rerun ablation
- **Run IV:** Species transfer experiment (human-only vs mouse-augmented vs mixed)
- **Run V:** Binary GOF/LOF only (drop No-net-effect class)
- **Run VI:** Add ESM-2 embeddings + AlphaMissense scores

---
## Run IV: mkdssp Real ASA + 80/20 Holdout Split
**Date:** 2026-08-13
**Config:** 797 variants, 66 features, 10 models, **80/20 stratified holdout** (NOT gene-grouped), 3 seeds (42/123/456), 20 Optuna trials, inner 5-fold
**Structural fix:** Real DSSP solvent accessibility via **mkdssp 4.0.4** (canonical Kabsch & Sander), run through WSL Ubuntu — replaces the simplified CIF-fallback DSSP that had ASA=0 everywhere
**CV change:** `train_test_split(test_size=0.2, stratify=y)` — a plain random split instead of leave-one-subunit-out grouping

Two changes are bundled here, so the score jump cannot be attributed to either one alone:
1. **mkdssp real ASA** — RSA (the #1 structural feature) now has 381 unique values instead of 1. ~88-97% of residues per structure now carry genuine ASA>0 (was ~0%).
2. **80/20 holdout** — trains ~638 / tests ~159 variants, with the same subunit appearing in both train and test (gene leakage).

In [ ]:
import pandas as pd
import numpy as np

# Run IV: 80/20 holdout + mkdssp real ASA (from results/comparison/comparison_holdout.csv)
run4_models = pd.DataFrame([
    {"Model": "CatBoost",            "Macro F1": 0.6624, "F1 ±": 0.0075, "MCC": 0.4953, "Bal Acc": 0.6761, "Accuracy": 0.6708},
    {"Model": "XGBoost",             "Macro F1": 0.6605, "F1 ±": 0.0241, "MCC": 0.4948, "Bal Acc": 0.6581, "Accuracy": 0.6854},
    {"Model": "Random Forest",       "Macro F1": 0.6578, "F1 ±": 0.0153, "MCC": 0.4863, "Bal Acc": 0.6632, "Accuracy": 0.6729},
    {"Model": "LightGBM",            "Macro F1": 0.6385, "F1 ±": 0.0169, "MCC": 0.4576, "Bal Acc": 0.6492, "Accuracy": 0.6500},
    {"Model": "MLP",                 "Macro F1": 0.6311, "F1 ±": 0.0491, "MCC": 0.4560, "Bal Acc": 0.6334, "Accuracy": 0.6562},
    {"Model": "Logistic Regression", "Macro F1": 0.6122, "F1 ±": 0.0449, "MCC": 0.4316, "Bal Acc": 0.6351, "Accuracy": 0.6188},
    {"Model": "SVM (Linear)",        "Macro F1": 0.6077, "F1 ±": 0.0353, "MCC": 0.4258, "Bal Acc": 0.6229, "Accuracy": 0.6208},
    {"Model": "SVM (RBF)",           "Macro F1": 0.5935, "F1 ±": 0.0208, "MCC": 0.3951, "Bal Acc": 0.5991, "Accuracy": 0.6146},
    {"Model": "KNN",                 "Macro F1": 0.5590, "F1 ±": 0.0219, "MCC": 0.3413, "Bal Acc": 0.5700, "Accuracy": 0.5708},
    {"Model": "Gaussian NB",         "Macro F1": 0.4414, "F1 ±": 0.0295, "MCC": 0.2498, "Bal Acc": 0.5137, "Accuracy": 0.4438},
])
run4_models = run4_models.sort_values("Macro F1", ascending=False).reset_index(drop=True)
run4_models.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
run4_models

### Run IV Key Takeaways

**⚠️ These scores are an upper bound (ceiling), not a generalization estimate.**

The +0.18 F1 jump over Run I (CatBoost 0.483 → 0.662) is driven almost entirely by the **80/20 holdout split**, not by the mkdssp feature fix:

1. **Gene leakage inflates every score.** With a random split, the same subunit (gene) appears in both train and test. The gene one-hot — the #1 most important feature per both Run I and Run II ablations — can be *memorized* per subunit. The model learns "CHRNA7 variants tend to be GOF" and re-applies that to CHRNA7 test variants it already saw in training.

2. **This is why the ranking is so stable** — CatBoost 0.662 vs Gaussian NB 0.441 still spans 0.22 F1, and the model ordering is identical to Run I. The split change lifts everyone up uniformly; it doesn't change *which* model wins.

3. **The mkdssp ASA fix is NOT isolated here.** To measure the structural-feature effect alone, we'd need to rerun the **gene-grouped CV** with real ASA and compare against Run II. (Run III already tested this with freesasa ASA and found **no improvement** — structural features are not the bottleneck; gene identity is.)

4. **For the thesis, use Run I/II numbers as the honest generalization estimate.** The holdout numbers are a sanity check ("can the model fit its own data?") but gene-grouped CV is the only defensible metric for a novel-subunit generalization claim.

### Run IV vs Run I (CatBoost)

| Metric | Run I (gene-grouped) | Run IV (80/20 holdout) | Δ |
|--------|---------------------|------------------------|---|
| Macro F1 | 0.483 | 0.662 | **+0.179** |
| MCC | 0.262 | 0.495 | +0.233 |
| Bal Acc | 0.506 | 0.676 | +0.170 |
| Accuracy | 0.529 | 0.671 | +0.142 |

### Next steps (revised)

- **Run V (priority):** Isolate the mkdssp effect — rerun gene-grouped CV with real ASA, compare to Run II to see if structural features now help.
- **Run VI:** Binary GOF/LOF only (drop No-net-effect class).
- **Run VII:** Species transfer experiment (human-only vs mouse-augmented vs mixed).
- **Run VIII:** Add ESM-2 embeddings + AlphaMissense scores.

---
## Run V: mkdssp Real ASA + Gene-Grouped CV (isolates the mkdssp effect)
**Date:** 2026-08-15
**Config:** 797 variants, 66 features, 10 models, **subunit CV** (leave-one-subunit-out `StratifiedGroupKFold` — no gene leakage), 3 seeds (42/123/456), 20 Optuna trials
**Change vs Run I/II:** real mkdssp 4.0.4 ASA (was simplified CIF-fallback with ASA=0 everywhere)
**Change vs Run IV:** gene-grouped split restored (was 80/20 holdout)

This is the apples-to-apples test for the mkdssp fix: same gene-grouped CV as Run I/II, but with real solvent accessibility. Any delta vs Run I is *purely* the structural-feature fix — no split confound. Total wall-clock 155 min.

In [ ]:
import pandas as pd
import numpy as np

# Run V: gene-grouped subunit CV + mkdssp real ASA (from results/comparison/comparison_subunit.csv)
run5_models = pd.DataFrame([
    {"Model": "CatBoost",            "Macro F1": 0.4891, "F1 ±": 0.0206, "MCC": 0.2743, "Bal Acc": 0.5189, "Accuracy": 0.5255},
    {"Model": "Random Forest",       "Macro F1": 0.4701, "F1 ±": 0.0235, "MCC": 0.2598, "Bal Acc": 0.4932, "Accuracy": 0.5194},
    {"Model": "LightGBM",            "Macro F1": 0.4663, "F1 ±": 0.0192, "MCC": 0.2514, "Bal Acc": 0.4939, "Accuracy": 0.5168},
    {"Model": "Logistic Regression", "Macro F1": 0.4611, "F1 ±": 0.0245, "MCC": 0.2460, "Bal Acc": 0.4928, "Accuracy": 0.4925},
    {"Model": "SVM (Linear)",        "Macro F1": 0.4464, "F1 ±": 0.0158, "MCC": 0.2195, "Bal Acc": 0.4743, "Accuracy": 0.4834},
    {"Model": "XGBoost",             "Macro F1": 0.4444, "F1 ±": 0.0335, "MCC": 0.2204, "Bal Acc": 0.4686, "Accuracy": 0.5028},
    {"Model": "SVM (RBF)",           "Macro F1": 0.4291, "F1 ±": 0.0191, "MCC": 0.1985, "Bal Acc": 0.4598, "Accuracy": 0.4782},
    {"Model": "KNN",                 "Macro F1": 0.4259, "F1 ±": 0.0143, "MCC": 0.1675, "Bal Acc": 0.4628, "Accuracy": 0.4598},
    {"Model": "MLP",                 "Macro F1": 0.4231, "F1 ±": 0.0213, "MCC": 0.1799, "Bal Acc": 0.4470, "Accuracy": 0.4688},
    {"Model": "Gaussian NB",         "Macro F1": 0.3867, "F1 ±": 0.0444, "MCC": 0.1837, "Bal Acc": 0.4456, "Accuracy": 0.4258},
])
run5_models = run5_models.sort_values("Macro F1", ascending=False).reset_index(drop=True)
run5_models.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
run5_models

### Run V Key Takeaways

**The mkdssp real-ASA fix does NOT move the needle.** This was the clean isolate test, and the answer is now definitive.

| Model | Run I (simplified DSSP) | Run V (mkdssp ASA) | Δ F1 |
|-------|------------------------|--------------------|------|
| CatBoost | 0.483 | 0.489 | +0.006 |
| Random Forest | 0.461 | 0.470 | +0.009 |
| LightGBM | 0.461 | 0.466 | +0.005 |
| Logistic Regression | 0.461 | 0.461 | 0.000 |
| SVM (RBF) | 0.459 | 0.429 | -0.030 |
| XGBoost | 0.451 | 0.444 | -0.007 |
| MLP | 0.451 | 0.423 | -0.028 |
| KNN | 0.440 | 0.426 | -0.014 |
| Gaussian NB | 0.368 | 0.387 | +0.019 |
| **Mean** | | | **-0.004** |

1. **Mean Δ ≈ -0.004 F1** — statistically indistinguishable from zero. Half the models improve a hair, half get a hair worse. This is noise, not signal.

2. **Confirms the Run II ablation.** Run II already flagged `structural_core` as neutral (+0.004 delta, "mixed: 5 help, 5 hurt"). Giving RSA its correct values doesn't change that — it stays neutral. The simplified DSSP was *not* the thing holding the model back.

3. **The bottleneck is elsewhere.** The inner-CV "Best inner F1" hits ~0.65 while held-out-subunit test F1 sits at ~0.49. That ~0.16 gap is the generalization ceiling from gene-grouped CV, and it's driven by gene identity (positional features, -0.042 in Run II), not by missing structural detail.

4. **Verdict for the thesis:** keep the mkdssp real ASA (it's scientifically defensible — RSA values are now physically meaningful), but do *not* expect it to improve GOF/LOF prediction. The structural-feature question is answered: it's a neutral feature group for this task. To move F1, the levers are elsewhere — binary GOF/LOF collapse (drop the ambiguous "No net effect" class), more data (mouse augmentation), or embeddings (ESM-2/AlphaMissense).

5. **The honest generalization number stays ~0.49.** Run IV's 0.66 was the leakage ceiling; Run V's 0.49 (CatBoost) is the real number to report.

### Run IV vs Run V — the split effect, quantified

| Model | Run IV (80/20 holdout) | Run V (gene-grouped) | Δ (leakage) |
|-------|------------------------|----------------------|-------------|
| CatBoost | 0.662 | 0.489 | +0.173 |
| Gaussian NB | 0.441 | 0.387 | +0.055 |
| XGBoost | 0.661 | 0.444 | +0.216 |

The 80/20 holdout inflates F1 by **+0.17 to +0.22** across models — this is the pure gene-leakage premium, now measured directly.

---
## Run VI: Binary GOF/LOF (drop No-net-effect class)
**Date:** 2026-08-16
**Config:** 594 variants (452 human / 87 rat / 55 mouse), 66 features, 10 models, **subunit CV** (gene-grouped, no leakage), 3 seeds (42/123/456), 20 Optuna trials
**Change vs Run V:** dropped the "No net effect" class → binary GOF (208) vs LOF (386). Same CV, same features, same seeds.
**Also fixed:** loader.py:112 NaN-species bug + dropped 9 rows with positions outside the human reference length (data-entry errors e.g. CHRNA1 p=1314, CHRNE p=-8).

This tests the Run II/V hypothesis that the ambiguous "No net effect" class was dragging down macro F1. Removing it isolates the GOF-vs-LOF signal — the scientifically interesting question, and the task MissION / VEP-ENaC actually solve.

In [ ]:
import pandas as pd
import numpy as np

# Run VI: binary GOF/LOF, gene-grouped subunit CV (from results/comparison/comparison_subunit_binary.csv)
run6_models = pd.DataFrame([
    {"Model": "Random Forest",       "Macro F1": 0.6445, "F1 ±": 0.0378, "MCC": 0.3268, "Bal Acc": 0.6640, "Accuracy": 0.6928},
    {"Model": "KNN",                 "Macro F1": 0.6335, "F1 ±": 0.0065, "MCC": 0.2997, "Bal Acc": 0.6547, "Accuracy": 0.6720},
    {"Model": "LightGBM",            "Macro F1": 0.6289, "F1 ±": 0.0150, "MCC": 0.2886, "Bal Acc": 0.6425, "Accuracy": 0.6675},
    {"Model": "CatBoost",            "Macro F1": 0.6289, "F1 ±": 0.0354, "MCC": 0.2949, "Bal Acc": 0.6486, "Accuracy": 0.6706},
    {"Model": "XGBoost",             "Macro F1": 0.6283, "F1 ±": 0.0415, "MCC": 0.3111, "Bal Acc": 0.6477, "Accuracy": 0.6850},
    {"Model": "SVM (RBF)",           "Macro F1": 0.5942, "F1 ±": 0.0502, "MCC": 0.2375, "Bal Acc": 0.6149, "Accuracy": 0.6435},
    {"Model": "MLP",                 "Macro F1": 0.5907, "F1 ±": 0.0064, "MCC": 0.2188, "Bal Acc": 0.6099, "Accuracy": 0.6339},
    {"Model": "Logistic Regression", "Macro F1": 0.5771, "F1 ±": 0.0281, "MCC": 0.2014, "Bal Acc": 0.5997, "Accuracy": 0.6173},
    {"Model": "SVM (Linear)",        "Macro F1": 0.5660, "F1 ±": 0.0727, "MCC": 0.1911, "Bal Acc": 0.5868, "Accuracy": 0.6067},
    {"Model": "Gaussian NB",         "Macro F1": 0.5006, "F1 ±": 0.0542, "MCC": 0.1461, "Bal Acc": 0.5758, "Accuracy": 0.5198},
])
run6_models = run6_models.sort_values("Macro F1", ascending=False).reset_index(drop=True)
run6_models.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
run6_models

### Run VI vs Run V — every model improves

Dropping the "No net effect" class lifts macro F1 for all 10 models by **+0.11 to +0.21** (mean +0.155).

⚠️ **Caveat:** macro F1 over 2 classes is not strictly comparable to macro F1 over 3 classes — the 3-class metric is penalized by an extra (harder) class, so part of the jump is arithmetic rather than signal. The clean read: **the NNE class was the single hardest class to predict and was suppressing the headline number.** Removing it surfaces the GOF/LOF skill that was always there underneath.

**Binary baselines:** always-predict-LOF (majority) = 0.394 macro F1; chance ≈ 0.50. Random Forest's 0.644 (MCC 0.327, Bal Acc 0.664) is real but modest signal — roughly in line with VEP-ENaC's binary ~0.6 reference.

### Key Takeaways

1. **Binary GOF/LOF is the right framing** — it's what MissION and VEP-ENaC actually solve, and it lifts every model by 0.11–0.21 macro F1. For the thesis, report binary as the primary task and keep 3-class as the harder secondary.

2. **Random Forest takes the crown** (0.644), with KNN/LightGBM/CatBoost/XGBoost clustered at ~0.63. Gaussian NB (0.50) is still worst — its naive independence assumption remains a poor fit.

3. **Model ranking is stable vs Run V** — RF, LightGBM, CatBoost, XGBoost stay near the top; Gaussian NB stays last. The class collapse lifts everyone uniformly without reshuffling the winner.

4. **⚠️ 142 rodent rows (87 rat + 55 mouse) are still contaminated** — they feed rodent residue numbers into human references/PDBs. That noise is still inside Run VI. After the cross-species remap (see Section: cross-species fix), the number may shift again (likely up).

5. **The remaining levers are data + chemistry, not structure.** mkdssp is confirmed neutral (Run V) and NNE is removed (Run VI). What's left: fix the broken `structural_nachr` features, add ThermoMPNN ΔΔG / ESM-2 embeddings, and correctly remap the rodent data.

In [ ]:
# Run VI (binary) vs Run V (3-class) — per-model macro F1 delta
run5_f1 = {"Logistic Regression": 0.4611, "SVM (RBF)": 0.4291, "SVM (Linear)": 0.4464,
           "Random Forest": 0.4701, "LightGBM": 0.4663, "KNN": 0.4259,
           "Gaussian NB": 0.3867, "MLP": 0.4231, "XGBoost": 0.4444, "CatBoost": 0.4891}
run6_f1 = {"Logistic Regression": 0.5771, "SVM (RBF)": 0.5942, "SVM (Linear)": 0.5660,
           "Random Forest": 0.6445, "LightGBM": 0.6289, "KNN": 0.6335,
           "Gaussian NB": 0.5006, "MLP": 0.5907, "XGBoost": 0.6283, "CatBoost": 0.6289}

run6_delta = pd.DataFrame([
    {"Model": m, "Run V (3-class)": run5_f1[m], "Run VI (binary)": run6_f1[m],
     "Δ F1": run6_f1[m] - run5_f1[m]} for m in run6_f1
]).sort_values("Run VI (binary)", ascending=False).reset_index(drop=True)
run6_delta["Δ F1"] = run6_delta["Δ F1"].map(lambda x: f"+{x:.3f}")
run6_delta

---
## Run VII: Cross-Species + Numbering Normalization (final_mapped.xlsx)
**Date:** 2026-08-19
**Config:** 717 variants (was 785), 66 features, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials
**Change vs Run V:** the "AA position" column re-anchored to the human precursor RefSeq numbering, and mouse/rat rows remapped onto the human reference.

### What was broken

`final.xlsx` stored positions in a **mixed numbering frame** — some genes in precursor (full-length) numbering, others in mature-protein numbering (signal peptide stripped, offset +20–33), plus a few data-entry typos. This was a *pre-existing* bug affecting roughly half the **human** data too, not just the rodent orthologs. Every position-keyed feature — normalized position, TM-helix, DSSP/RSA, B-factor, pore distance — was silently ~20–33 residues off for those rows.

### The fix

`scripts/normalize_positions.py` anchors each row back onto the reference using its **wildtype amino acid** (the one reliable ground truth papers report). Resolution order: precursor → mature (+signal peptide) → typo (±5) → isoform (+15..40) → drop.

| Outcome | Rows |
|---------|------|
| Resolved to human precursor numbering | 755 / 841 (89.8%) |
| Dropped (position blanked) | 86 / 841 (10.2%) |

Status breakdown: mature 352, precursor 161, mature+ortholog 115, precursor+ortholog 62, typo 19, isoform+ortholog 17, typo+ortholog 17, isoform 12. Dropped: 73 ambiguous + 11 no_match + 1 no_reference + 1 missing.

| Dataset | Variants | Human / Rat / Mouse |
|---------|----------|---------------------|
| Baseline (`final.xlsx`) | 785 | 533 / 174 / 78 |
| Mapped (`final_mapped.xlsx`) | 717 | 507 / 138 / 72 |

Net −68 variants (8.7%), but every surviving row now carries a correct precursor position.

In [ ]:
import pandas as pd
import numpy as np

# Run VII vs Run V: numbering normalization, gene-grouped subunit CV
# Baseline = results/comparison/comparison_subunit.csv (Run V)
# Mapped   = results/comparison/comparison_subunit_mapped.csv (Run VII)
run5 = {"Logistic Regression": 0.4611, "SVM (RBF)": 0.4291, "SVM (Linear)": 0.4464,
        "Random Forest": 0.4701, "LightGBM": 0.4663, "KNN": 0.4259,
        "Gaussian NB": 0.3867, "MLP": 0.4231, "XGBoost": 0.4444, "CatBoost": 0.4891}
run7 = {"Logistic Regression": 0.4534, "SVM (RBF)": 0.4370, "SVM (Linear)": 0.4472,
        "Random Forest": 0.4834, "LightGBM": 0.4628, "KNN": 0.4472,
        "Gaussian NB": 0.3768, "MLP": 0.4578, "XGBoost": 0.4599, "CatBoost": 0.4531}

run7_delta = pd.DataFrame([
    {"Model": m, "Run V (unmapped)": run5[m], "Run VII (mapped)": run7[m],
     "Δ F1": run7[m] - run5[m]} for m in run5
]).sort_values("Run VII (mapped)", ascending=False).reset_index(drop=True)
run7_delta["Δ F1"] = run7_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
run7_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — Run V: %.4f | Run VII: %.4f | Δ: %+.4f" % (
    np.mean(list(run5.values())), np.mean(list(run7.values())),
    np.mean(list(run7.values())) - np.mean(list(run5.values()))))
run7_delta

### Run VII Key Takeaways

**The normalization is F1-neutral (mean Δ = +0.004).** The hypothesis that fixing the residue positions would lift the score did not hold — 7 of 10 models moved within ±0.02 (noise), and the mean is essentially unchanged. The honest headline: the mapping does **not** improve generalization macro F1.

1. **CatBoost collapsed (−0.036).** The Run V champion (0.489) fell to 0.453. Its prior edge was partly an artifact of the noisy positions (and a favorable fold split) rather than robust signal. With corrected positions *and* 68 fewer training rows, it falls back into the pack.

2. **Random Forest becomes the new best (0.483).** RF (+0.013), XGBoost (+0.016), MLP (+0.035), and KNN (+0.021) all improved. The leaderboard is now flatter and RF — already the binary-mode winner in Run VI — is also the 3-class winner.

3. **This is exactly what the Run II/V ablations predicted.** Position-dependent features (normalized position, structural) are near-neutral (structural_core +0.004, structural_nachr −0.009). The model's signal is dominated by **gene one-hot + physicochemical properties**, neither of which depends on the residue position being correct. Fixing the input to already-neutral features cannot move the output.

4. **Keep the mapping anyway.** The data is now scientifically correct: a variant reported as "mature residue 50" is stored as its true precursor position, and every structural feature now reads the right residue. For the thesis this is non-negotiable correctness — even though it doesn't change the headline F1.

5. **The real ceiling is still gene-level generalization.** The inner-CV "best inner F1" (~0.65) vs held-out-subunit test F1 (~0.45–0.48) gap is unchanged. The levers that actually matter remain: more data (cross-species augmentation), embeddings (ESM-2/AlphaMissense/ThermoMPNN), or reframing the task — not residue numbering.

### Run VII vs Run V — the honest read

| Metric | Run V (unmapped) | Run VII (mapped) | Δ |
|--------|-----------------|------------------|---|
| Mean macro F1 (10 models) | 0.444 | 0.448 | +0.004 |
| Best model F1 | 0.489 (CatBoost) | 0.483 (Random Forest) | −0.006 |
| Data integrity | mixed precursor/mature frame | all precursor, WT-AA verified | ✅ fixed |

### Run VII — Binary GOF/LOF: same normalization, same (null) result
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, subunit CV, 3 seeds (42/123/456), 30 Optuna trials — identical settings to Run VI but on `final_mapped.xlsx`.
**Change vs Run VI:** the same numbering normalization as the 3-class run above (plus the loader's NaN-species + out-of-range fixes already present in Run VI).

In [ ]:
import pandas as pd
import numpy as np

# Binary GOF/LOF: numbering normalization (mapped vs Run VI baseline)
run6_bin = {"Logistic Regression": 0.5771, "SVM (RBF)": 0.5942, "SVM (Linear)": 0.5660,
            "Random Forest": 0.6445, "LightGBM": 0.6289, "KNN": 0.6335,
            "Gaussian NB": 0.5006, "MLP": 0.5907, "XGBoost": 0.6283, "CatBoost": 0.6289}
run7_bin = {"Logistic Regression": 0.6070, "SVM (RBF)": 0.5611, "SVM (Linear)": 0.5924,
            "Random Forest": 0.6291, "LightGBM": 0.6020, "KNN": 0.6012,
            "Gaussian NB": 0.5497, "MLP": 0.5867, "XGBoost": 0.6411, "CatBoost": 0.6303}

bin_delta = pd.DataFrame([
    {"Model": m, "Run VI (unmapped)": run6_bin[m], "Run VII (mapped)": run7_bin[m],
     "Δ F1": run7_bin[m] - run6_bin[m]} for m in run6_bin
]).sort_values("Run VII (mapped)", ascending=False).reset_index(drop=True)
bin_delta["Δ F1"] = bin_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
bin_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — Run VI: %.4f | Run VII (binary): %.4f | Δ: %+.4f" % (
    np.mean(list(run6_bin.values())), np.mean(list(run7_bin.values())),
    np.mean(list(run7_bin.values())) - np.mean(list(run6_bin.values()))))
bin_delta

### Run VII (binary) Key Takeaways

**Binary is also a wash — mean Δ = +0.001.** The numbering normalization is F1-neutral in *both* framings. This closes the loop on Run VI's open question: it predicted "the number may shift again (likely up)" after the rodent remap; in fact it shifted by ~nothing.

1. **XGBoost takes the binary crown (0.641),** overtaking Random Forest (0.644 → 0.629). The two are statistically indistinguishable (Δ +0.013 vs −0.015, both within the ±0.03–0.05 fold noise).

2. **The top binary models dipped, the mid-tier rose.** RF −0.015, LightGBM −0.027, KNN −0.032; but LR +0.030, SVM-Linear +0.026, Gaussian NB +0.049. Net: flat. Removing the contaminated rodent positions didn't remove any *usable* signal — the model was already discounting those rows.

3. **Gaussian NB's +0.049 jump is the largest single move** but irrelevant — it remains the worst model (0.550). Its naive independence assumption is the bottleneck, not the positions.

4. **Conclusion across both framings:** residue-numbering correctness is a **data-integrity** win, not a **predictive** win. The F1 ceiling is set by gene-level generalization (the ~0.65 inner-CV → ~0.48–0.64 test gap) and by data volume — not by where the residues are numbered. To move the number, the levers are more data (mouse/rat augmentation at scale), embeddings (ESM-2 / AlphaMissense / ThermoMPNN ΔΔG), or a different feature set.

---
## Run VIII: + Position Conservation & ESM-2 Zero-Shot Features (66 → 71 features)
**Date:** 2026-08-21
**Config:** 717 variants, **71 features**, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials
**Change vs Run VII:** two new feature groups added — (1) **position-specific evolutionary conservation** from the mouse/rat ortholog alignments, and (2) **ESM-2 protein language model** zero-shot variant-effect scores.

### The two new features (and why they target the real bottleneck)

Run VII concluded the F1 ceiling is set by *gene-level generalization*, and named "embeddings + more data" as the levers that would actually move the number. These two features attack that ceiling directly — without touching the data pool (still 717 variants):

1. **Conservation (3 features: `conservation_wt`, `conservation_mt`, `conservation_delta`).** BLOSUM62/Grantham measure *how big* the amino-acid change is — position-independent. Conservation instead measures *how constrained this exact residue* is: `conservation_wt` = fraction of aligned ortholog columns (mouse/rat/human) carrying the wildtype AA at that position, `conservation_mt` = the same for the mutant, `conservation_delta` = the drop. Built entirely from the precomputed ortholog mappings (`mapping/{mouse,rat}_to_human.csv`) — **no new downloads**. Mean `conservation_wt` = 0.936 (std 0.181): nAChR residues are, on average, strongly conserved.

2. **ESM-2 (2 features: `esm2_wt_logprob`, `esm2_effect`).** The masked-marginal zero-shot score (Meier et al. 2021): mask the variant position, run ESM-2 (`esm2_t30_150M_UR50D`), and read `log P(mutant | context) − log P(wildtype | context)`. `esm2_wt_logprob` is an ESM-based conservation proxy (how "surprising" the wildtype residue is); `esm2_effect` is the zero-shot mutation effect (negative = disfavoured). Range [−12.4, +6.4], mean −3.1. CPU-friendly 150M checkpoint, loaded lazily and cached.


In [ ]:
import pandas as pd
import numpy as np

# Run VIII vs Run VII (3-class, gene-grouped subunit CV)
# Baseline = results/comparison/comparison_subunit_mapped.csv   (Run VII,  66 features)
# New      = results/comparison/comparison_subunit_features_v2.csv (Run VIII, 71 features)
run7 = {"Logistic Regression": 0.4534, "SVM (RBF)": 0.4370, "SVM (Linear)": 0.4472,
        "Random Forest": 0.4834, "LightGBM": 0.4628, "KNN": 0.4472,
        "Gaussian NB": 0.3768, "MLP": 0.4578, "XGBoost": 0.4599, "CatBoost": 0.4531}
run8 = {"Logistic Regression": 0.4834, "SVM (RBF)": 0.4099, "SVM (Linear)": 0.4745,
        "Random Forest": 0.5012, "LightGBM": 0.4750, "KNN": 0.4651,
        "Gaussian NB": 0.4586, "MLP": 0.4669, "XGBoost": 0.4695, "CatBoost": 0.5081}

run8_delta = pd.DataFrame([
    {"Model": m, "Run VII (66 feats)": run7[m], "Run VIII (71 feats)": run8[m],
     "Δ F1": run8[m] - run7[m]} for m in run7
]).sort_values("Run VIII (71 feats)", ascending=False).reset_index(drop=True)
run8_delta["Δ F1"] = run8_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
run8_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — Run VII: %.4f | Run VIII: %.4f | Δ: %+.4f" % (
    np.mean(list(run7.values())), np.mean(list(run8.values())),
    np.mean(list(run8.values())) - np.mean(list(run7.values()))))
run8_delta


### Run VIII (3-class) Key Takeaways

**The features work — mean Δ = +0.023, 9/10 models improve, new best CatBoost 0.508.** Unlike the numbering normalization (Run VII, F1-neutral), adding conservation + ESM-2 *did* move the number. This is the first feature-driven improvement of the whole project — every prior run moved F1 through data fixes or task reframing, not signal.

1. **New champion: CatBoost 0.508 (+0.055).** It jumps from 0.453 (7th) to 0.508 (1st), overtaking Random Forest (0.483 → 0.501). The single biggest move in the table — conservation + ESM-2 carry signal that ordered boosting can exploit.

2. **The weak models surged, the field compressed.** Gaussian NB +0.082 (its best-ever, though still last), LR +0.030, SVM-Linear +0.027, CatBoost +0.055. The additions inject *positional/constraint* signal that linear and naive models previously couldn't see through the gene one-hot + physicochemical features.

3. **One regression: SVM-RBF (−0.027),** and its fold std roughly doubled to 0.045 (highest in the table). The ESM-2 log-probs are heavy-tailed (−12…+6, clustered near −3). RBF does get `RobustScaler` (median/IQR) via `needs_scaling=True`, but that doesn't fully tame a heavy tail — the RBF kernel's squared distance can still be dominated by those tail values. Likely a scaling/tail interaction, not a signal problem. Worth checking by winsorizing/clipping or standardizing the ESM columns, or dropping them for the RBF model alone.

4. **The ceiling is budging, but only just.** 0.508 is still far below the inner-CV ~0.65 and not yet usable. Conservation + ESM-2 are the *cheap* version of the "embeddings + more data" lever Run VII named — and they now confirm that lever is the right direction. Next: full embeddings (ESM-2 650M, AlphaMissense, ThermoMPNN ΔΔG) and scaling the mouse/rat augmentation.

### Run VIII vs Run VII — 3-class

| Metric | Run VII (66 feats) | Run VIII (71 feats) | Δ |
|--------|-------------------|---------------------|---|
| Mean macro F1 (10 models) | 0.448 | 0.471 | +0.023 |
| Best model F1 | 0.483 (Random Forest) | 0.508 (CatBoost) | +0.025 |
| Models improved | — | 9 / 10 | ✅ |

*(Binary subunit, 3-class holdout, and binary holdout results follow below.)*


### Run VIII — Binary GOF/LOF: the same features do *not* help the direction task
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features**, subunit CV, 3 seeds (42/123/456), 30 Optuna trials — identical settings to Run VII binary, plus conservation + ESM-2.
**Change vs Run VII binary:** the two new feature groups (conservation + ESM-2), same as the 3-class run above.


In [ ]:
import pandas as pd
import numpy as np

# Binary GOF/LOF: conservation + ESM-2 (features_v2 vs mapped baseline)
# Baseline = results/comparison/comparison_subunit_binary_mapped.csv   (Run VII, 66 features)
# New      = results/comparison/comparison_subunit_binary_features_v2.csv (Run VIII, 71 features)
run7_bin = {"Logistic Regression": 0.6070, "SVM (RBF)": 0.5611, "SVM (Linear)": 0.5924,
            "Random Forest": 0.6291, "LightGBM": 0.6020, "KNN": 0.6012,
            "Gaussian NB": 0.5497, "MLP": 0.5867, "XGBoost": 0.6411, "CatBoost": 0.6303}
run8_bin = {"Logistic Regression": 0.5978, "SVM (RBF)": 0.5675, "SVM (Linear)": 0.5793,
            "Random Forest": 0.6469, "LightGBM": 0.6169, "KNN": 0.5858,
            "Gaussian NB": 0.5853, "MLP": 0.5955, "XGBoost": 0.6245, "CatBoost": 0.6366}

bin_delta = pd.DataFrame([
    {"Model": m, "Run VII bin (66)": run7_bin[m], "Run VIII bin (71)": run8_bin[m],
     "Δ F1": run8_bin[m] - run7_bin[m]} for m in run7_bin
]).sort_values("Run VIII bin (71)", ascending=False).reset_index(drop=True)
bin_delta["Δ F1"] = bin_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
bin_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — Run VII bin: %.4f | Run VIII bin: %.4f | Δ: %+.4f" % (
    np.mean(list(run7_bin.values())), np.mean(list(run8_bin.values())),
    np.mean(list(run8_bin.values())) - np.mean(list(run7_bin.values()))))
bin_delta


### Run VIII (binary) Key Takeaways

**Binary is flat — mean Δ = +0.004, only 6/10 models improve.** The same conservation + ESM-2 features that lifted 3-class by +0.023 do essentially *nothing* for GOF/LOF direction. This asymmetry is itself the finding.

1. **The features separate "effect vs no-effect", not "GOF vs LOF".** Ortholog conservation and ESM-2's masked-marginal score both answer *"how much does this mutation perturb the protein?"* — which maps cleanly onto the 3-class task's No-net-effect vs damaging axis. But a GOF mutation and a LOF mutation are **both** "damaging/constrained" from that viewpoint, so the features are blind to the *direction*. The 3-class gain was almost entirely the model getting better at spotting No-net-effect; the GOF↔LOF split it already could make is unchanged.

2. **Best binary model shifts within noise:** Random Forest 0.647 nudges past XGBoost's 0.641 baseline — the two have swapped places across every run and are statistically indistinguishable (±0.03–0.05 fold noise).

3. **This is the clearest evidence yet that direction-of-effect is a distinct, harder problem.** Generic pathogenicity/constraint features (ESM-2, conservation — and by extension AlphaMissense, PolyPhen, CADD) can triage *whether* a variant matters but not *which way* it pushes the receptor. That is precisely the gap the project exists to fill — and why a nAChR-specific GOF/LOF model is not just a re-labelled pathogenicity classifier.

### Run VIII vs Run VII — binary

| Metric | Run VII bin (66) | Run VIII bin (71) | Δ |
|--------|-----------------|-------------------|-----|
| Mean macro F1 (10 models) | 0.600 | 0.604 | +0.004 |
| Best model F1 | 0.641 (XGBoost) | 0.647 (Random Forest) | +0.006 |
| Models improved | — | 6 / 10 | ~noise |

*(Holdout results follow below.)*


### Run VIII — 3-class Holdout: confounded comparison, but the direction is telling
**Config:** 717 mapped variants, **71 features**, 3-class, **holdout (random train/test split — gene-leaky)**, 3 seeds (42/123/456), 30 Optuna trials.
**⚠️ Caveat:** the holdout baseline (`comparison_holdout.csv`, 2026-08-14) was generated **before** the numbering normalization (unmapped, 785 variants), and there is no mapped 66-feature holdout run to compare against. This table therefore conflates **two** changes — the normalization (−68 rows) *and* the new features. The subunit CV above is the controlled comparison; read this one directionally, not as a clean ablation.


In [ ]:
import pandas as pd
import numpy as np

# 3-class holdout: baseline is PRE-normalization (unmapped), new is mapped + 71 features.
# Baseline = results/comparison/comparison_holdout.csv          (66 feats, unmapped, 785 vars)
# New      = results/comparison/comparison_holdout_features_v2.csv (71 feats, mapped, 717 vars)
ho_base = {"Random Forest": 0.6578, "CatBoost": 0.6624, "XGBoost": 0.6605,
           "LightGBM": 0.6385, "MLP": 0.6311, "Logistic Regression": 0.6122,
           "SVM (Linear)": 0.6077, "SVM (RBF)": 0.5935, "KNN": 0.5590,
           "Gaussian NB": 0.4414}
ho_new = {"Random Forest": 0.6411, "CatBoost": 0.6409, "XGBoost": 0.6042,
          "LightGBM": 0.6059, "MLP": 0.5949, "Logistic Regression": 0.6159,
          "SVM (Linear)": 0.6152, "SVM (RBF)": 0.6079, "KNN": 0.5518,
          "Gaussian NB": 0.5093}

ho_delta = pd.DataFrame([
    {"Model": m, "Base holdout (66, unmapped)": ho_base[m],
     "Holdout + features (71, mapped)": ho_new[m], "Δ F1": ho_new[m] - ho_base[m]}
    for m in ho_base
]).sort_values("Holdout + features (71, mapped)", ascending=False).reset_index(drop=True)
ho_delta["Δ F1"] = ho_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
ho_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — base holdout: %.4f | holdout + features: %.4f | Δ: %+.4f" % (
    np.mean(list(ho_base.values())), np.mean(list(ho_new.values())),
    np.mean(list(ho_new.values())) - np.mean(list(ho_base.values()))))
ho_delta


### Run VIII (holdout) Key Takeaways

**Holdout mean −0.008, and the pattern is a mirror-image of subunit.** Only 4/10 models improve, and the *trees* — not the weak models — are the ones that dropped. Read with the confound caveat above; the interpretation is still useful:

1. **Trees dropped, weak models rose.** XGBoost −0.056, MLP −0.036, LightGBM −0.033, CatBoost −0.022, RF −0.017; but Gaussian NB +0.068, SVM-RBF +0.014, SVM-Linear +0.008. The *"features help the weak models"* signature from subunit CV is still visible — it's just swamped on the tree side by the other change (normalization).

2. **Why the trees dropped: leakage removal, not signal loss.** In a gene-leaky random split, the pre-normalization tree models were exploiting the *noisy residue positions* as a free gene-identity proxy (a variant's mis-numbered position partially encodes which gene it came from). The normalization stripped that leak, so the leaky holdout score falls back toward the honest number. This is the same leakage Run VII documented, just seen from the other side.

3. **This is exactly why the project uses gene-grouped CV.** The holdout (~0.60–0.66) is the optimistic, VEP-ENaC-style number; the subunit CV (~0.47–0.51) is the honest one. The features' real effect (+0.023 on subunit) is invisible in this confounded holdout, which is *fine* — the holdout was never the metric that matters.


### Run VIII — Binary Holdout: leaky ceiling, no baseline to compare
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features**, **holdout (random split, gene-leaky)**, 3 seeds (42/123/456), 30 Optuna trials.
**Note:** there is **no** pre-normalization binary holdout baseline (this mode was never run on the old data), so this is absolute numbers only — the leaky/optimistic ceiling for the binary task.


In [ ]:
import pandas as pd

# Binary holdout (random split, gene-leaky) — absolute macro F1, no baseline exists.
# Source: results/comparison/comparison_holdout_binary_features_v2.csv (71 feats, mapped)
bin_ho = {"KNN": 0.7599, "XGBoost": 0.7328, "LightGBM": 0.7294, "SVM (RBF)": 0.7282,
          "CatBoost": 0.7081, "Random Forest": 0.7042, "MLP": 0.7039,
          "SVM (Linear)": 0.6964, "Logistic Regression": 0.6652, "Gaussian NB": 0.6180}
bin_ho_df = pd.DataFrame([{"Model": m, "Binary holdout F1 (leaky)": f} for m, f in bin_ho.items()])
bin_ho_df = bin_ho_df.sort_values("Binary holdout F1 (leaky)", ascending=False).reset_index(drop=True)
bin_ho_df.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean binary holdout F1: %.4f" % bin_ho_df["Binary holdout F1 (leaky)"].mean())
bin_ho_df


### Run VIII (binary holdout) Key Takeaways

**Leaky binary ceiling ≈ 0.70 (best KNN 0.760), vs honest binary subunit ≈ 0.60.** The ~0.10 gap is the same gene-leakage Run VII documented: a random split lets the model score "same gene → same effect" on its own test rows.

1. **KNN winning (0.760) is a leaky-split artifact**, not a real champion — nearest neighbours literally find same-gene variants in the train fold. XGBoost (0.733), LightGBM (0.729), SVM-RBF (0.728) are the meaningful leaders here, and they cluster tightly.

2. **No before/after is possible** (this mode was never run pre-normalization), so this section is a ceiling reference, not a comparison. It is the number to quote if anyone asks "what's the optimistic (VEP-ENaC-style) score?" — and the reason the project's headline metric stays on the honest subunit CV.


---
## Run VIII — Overall: a real, partial win for the honest metric

| Framing | CV | Clean? | Mean Δ F1 | Best (Run VIII) |
|---|---|---|---|---|
| 3-class | subunit (honest) | ✅ | **+0.023** | CatBoost 0.508 |
| Binary GOF/LOF | subunit (honest) | ✅ | +0.004 | Random Forest 0.647 |
| 3-class | holdout (leaky) | ⚠️ confounded | −0.008 | Random Forest 0.641 |
| Binary GOF/LOF | holdout (leaky) | — (no baseline) | — | KNN 0.760 |

**The single finding.** Conservation + ESM-2 lift the honest 3-class macro-F1 by **+0.023 (9/10 models)** — the first feature-driven improvement of the entire project — but leave binary GOF/LOF unchanged (+0.004). The reason is coherent: both features answer *"how much does this mutation perturb the protein?"*, which cleanly separates **effect vs no-effect** (the 3-class No-net-effect axis) but is blind to **GOF vs LOF direction** (both read as "damaging"). That asymmetry is direct experimental evidence that direction-of-effect is orthogonal to pathogenicity — and that generic scores (ESM-2, conservation, and by extension AlphaMissense / PolyPhen / CADD) cannot substitute for a nAChR-specific GOF/LOF model.

**Housekeeping.** (1) SVM-RBF regressed in 3-class (−0.027, std 0.045) — a likely heavy-tail/scale interaction with the ESM-2 log-probs; worth winsorizing or standardizing the ESM columns. (2) The 3-class holdout comparison is confounded (its baseline predates the normalization) — the subunit CV is the controlled number.

**Next levers (now better-motivated).** Full embeddings (ESM-2 650M, AlphaMissense, ThermoMPNN ΔΔG) and cross-species augmentation at scale — both target the same *effect-vs-no-effect* axis these two cheap features already moved. Crucially, neither will close the *direction* gap: that needs nAChR-specific structure/mechanism signal (open-vs-closed conformational state, ligand/interface context), which is the project's actual contribution.


---
## Run IX: ESM-2 150M → 650M (null result — bigger model buys nothing)
**Date:** 2026-08-21
**Config:** 717 variants, 71 features, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials — identical to Run VIII, but the two ESM-2 zero-shot features are recomputed from `esm2_t33_650M_UR50D` (2.6 GB) instead of `esm2_t30_150M_UR50D`.
**Change vs Run VIII:** only the ESM model size changed (150M → 650M). Same masked-marginal method, same 2 features, same 71 total. A clean A/B.


In [ ]:
import pandas as pd
import numpy as np

# Run IX vs Run VIII (3-class, gene-grouped subunit CV): ESM-2 650M vs 150M
# 150M = results/comparison/comparison_subunit_features_v2.csv (Run VIII)
# 650M = results/comparison/comparison_subunit_650m.csv       (Run IX)
run8 = {"Logistic Regression": 0.4834, "SVM (RBF)": 0.4099, "SVM (Linear)": 0.4745,
        "Random Forest": 0.5012, "LightGBM": 0.4750, "KNN": 0.4651,
        "Gaussian NB": 0.4586, "MLP": 0.4669, "XGBoost": 0.4695, "CatBoost": 0.5081}
run9 = {"Logistic Regression": 0.4561, "SVM (RBF)": 0.4339, "SVM (Linear)": 0.4647,
        "Random Forest": 0.5102, "LightGBM": 0.4744, "KNN": 0.4664,
        "Gaussian NB": 0.4498, "MLP": 0.4457, "XGBoost": 0.4650, "CatBoost": 0.4830}

run9_delta = pd.DataFrame([
    {"Model": m, "150M (Run VIII)": run8[m], "650M (Run IX)": run9[m],
     "Δ F1": run9[m] - run8[m]} for m in run8
]).sort_values("650M (Run IX)", ascending=False).reset_index(drop=True)
run9_delta["Δ F1"] = run9_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
run9_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — 150M: %.4f | 650M: %.4f | Δ: %+.4f" % (
    np.mean(list(run8.values())), np.mean(list(run9.values())),
    np.mean(list(run9.values())) - np.mean(list(run8.values()))))
run9_delta


### Run IX (3-class) Key Takeaways

**650M ≈ 150M — mean Δ = −0.006, 3/10 models improve, best model unchanged (0.510 RF vs 0.508 CatBoost).** Scaling the ESM-2 checkpoint 4.3× does not move the honest metric. This is a clean A/B: only the ESM model size changed.

1. **The per-model swings are pure noise.** CatBoost −0.025, LR −0.027, MLP −0.021, but SVM-RBF +0.024, RF +0.009. Every one sits within the ±0.02–0.03 fold/seed std we see run-to-run; the −0.006 mean is indistinguishable from zero.

2. **Zero-shot score *quality* was never the bottleneck.** 150M and 650M rank "is this mutation damaging?" near-identically — the extra parameters refine fine-grained likelihoods that don't change the coarse GOF/LOF/no-net signal. This closes the "bigger LM = better features" hypothesis: the lever is the **axis** (effect-vs-no-effect vs direction) and **data volume**, not model capacity.

3. **Keep 150M.** It's ~4× faster on extraction (~17 min vs ~1.3 h) and equally good; the 650M end-to-end was ~2× slower (~5 h vs ~2 h for 3-class subunit). Further embedding work should chase *different* signals (structure-based, ΔΔG) — not bigger LMs.

### Run IX vs Run VIII — 3-class

| Metric | 150M (Run VIII) | 650M (Run IX) | Δ |
|--------|----------------|---------------|---|
| Mean macro F1 (10 models) | 0.471 | 0.465 | −0.006 |
| Best model F1 | 0.508 (CatBoost) | 0.510 (Random Forest) | +0.002 |
| Models improved | — | 3 / 10 | noise |


### Run IX — Binary GOF/LOF: 650M is flat too (Δ −0.0002)
**Config:** 553 mapped variants (GOF 190 / LOF 363), binary task, **71 features** with the two ESM-2 zero-shot features recomputed from `esm2_t33_650M_UR50D`, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials — identical to Run VIII binary.
**Change vs Run VIII binary:** only the ESM-2 checkpoint changed (150M → 650M). A clean A/B on the direction task.


In [ ]:
import pandas as pd
import numpy as np

# Run IX vs Run VIII (binary GOF/LOF, gene-grouped subunit CV): ESM-2 650M vs 150M
# 150M = results/comparison/comparison_subunit_binary_features_v2.csv (Run VIII)
# 650M = results/comparison/comparison_subunit_binary_650m.csv       (Run IX)
run8_bin = {"Logistic Regression": 0.5978, "SVM (RBF)": 0.5675, "SVM (Linear)": 0.5793,
            "Random Forest": 0.6469, "LightGBM": 0.6169, "KNN": 0.5858,
            "Gaussian NB": 0.5853, "MLP": 0.5955, "XGBoost": 0.6245, "CatBoost": 0.6366}
run9_bin = {"Logistic Regression": 0.5997, "SVM (RBF)": 0.5682, "SVM (Linear)": 0.6099,
            "Random Forest": 0.6387, "LightGBM": 0.6049, "KNN": 0.5967,
            "Gaussian NB": 0.5861, "MLP": 0.6043, "XGBoost": 0.6243, "CatBoost": 0.6015}

bin_delta = pd.DataFrame([
    {"Model": m, "150M (Run VIII)": run8_bin[m], "650M (Run IX)": run9_bin[m],
     "Δ F1": run9_bin[m] - run8_bin[m]} for m in run8_bin
]).sort_values("650M (Run IX)", ascending=False).reset_index(drop=True)
bin_delta["Δ F1"] = bin_delta["Δ F1"].map(lambda x: f"{x:+.3f}")
bin_delta.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Mean macro F1 — 150M: %.4f | 650M: %.4f | Δ: %+.4f" % (
    np.mean(list(run8_bin.values())), np.mean(list(run9_bin.values())),
    np.mean(list(run9_bin.values())) - np.mean(list(run8_bin.values()))))
bin_delta


### Run IX (binary) Key Takeaways

**Binary is flat too — mean Δ = −0.0002, 6/10 models improve, best model (RF) drops 0.647 → 0.639.** The 650M checkpoint is null on *both* framings. This completes the A/B: ESM model size is irrelevant to the honest metric.

1. **The wins are small, the losses are lumpy.** SVM-Linear +0.031, KNN +0.011, MLP +0.009 — all within fold noise. But CatBoost −0.035 and LightGBM −0.012 drop hard, mirroring the 3-class result (CatBoost −0.025). The model that rode the new features hardest in Run VIII (CatBoost) is the most sensitive to the checkpoint swap, which suggests part of its Run VIII edge was tuned to the 150M log-probs.

2. **Best binary model is unchanged in identity, nudged down in value.** Random Forest stays on top (0.647 → 0.639); XGBoost (0.624) and CatBoost (0.637 → 0.601) reorder beneath it. Nothing crosses the ±0.03–0.05 fold-noise band.

3. **Verdict on the "bigger LM" lever: a dead end.** 150M and 650M produce the same GOF/LOF/no-net signal — the zero-shot perturbation score was already saturated at 150M. The remaining levers are unchanged and now the only ones left: **data** (cross-species augmentation at scale), **axis** (effect-vs-no-effect scores like ΔΔG), and **nAChR mechanism** (open/closed conformation, ligand/interface context) for the direction gap that no amount of LM scale touches.

### Run IX vs Run VIII — binary

| Metric | 150M (Run VIII) | 650M (Run IX) | Δ |
|--------|----------------|---------------|---|
| Mean macro F1 (10 models) | 0.604 | 0.603 | −0.000 |
| Best model F1 | 0.647 (Random Forest) | 0.639 (Random Forest) | −0.008 |
| Models improved | — | 6 / 10 | noise |


---

## Run IX — Overall: ESM-2 650M is a null on both tasks

| Framing | CV | Mean Δ F1 | Best (150M → 650M) |
|---|---|---|---|
| 3-class | subunit (honest) | −0.006 | CatBoost 0.508 → RF 0.510 |
| Binary GOF/LOF | subunit (honest) | −0.000 | RF 0.647 → RF 0.639 |

**The single finding.** Scaling ESM-2 from 150M to 650M (4.3× parameters, ~2× slower end-to-end, 2.6 GB weights) changes nothing on the honest gene-grouped metric — 3-class −0.006, binary −0.000. The zero-shot "how much does this perturb the protein?" signal saturates at 150M; the extra parameters refine fine-grained likelihoods that don't affect the coarse functional-direction output.

**What this means for the thesis.** Run VIII's +0.023 (3-class) was a *feature-type* win (conservation + ESM-2), not a *model-capacity* win. Run IX closes the capacity question: bigger LMs are not the lever. The levers that remain are **data volume** (mouse/rat augmentation — the biggest headroom), **axis** (ΔΔG and other effect-vs-no-effect scores for cheap 3-class gains), and **nAChR-specific mechanism** (open/closed conformation, ligand/interface context) — the only signal that can close the GOF/LOF *direction* gap, which generic scores are provably blind to (Run VIII).

**Housekeeping.** Revert `embeddings.py` to `esm2_t30_150M_UR50D` — it's ~4× faster to extract and equally good. Cache note: the feature cache keys on the data hash only, so any future ESM-model swap must `python -m vep_nachr2.training.runner clean` first (already done for this run).


# Run X: + AlphaMissense (71 → 72 features) — null result; confirms direction ⊥ pathogenicity

**Date:** 2026-08-22
**Config:** 717 variants, **72 features**, 10 models, subunit CV (gene-grouped, no leakage), 3 seeds (42/123/456), 30 Optuna trials.
**Change vs Run VIII:** one new feature group — **AlphaMissense** (`alphamissense_pathogenicity`, 1 feature): DeepMind's per-substitution pathogenicity score (Cheng et al. 2023), keyed by UniProt accession. ESM-2 reverted to 150M per Run IX.

## The feature + a CHRNA1 numbering fix

AlphaMissense scores *pathogenicity* (0–1, "how damaging is this substitution?") at each UniProt position. Two implementation notes:

1. **RefSeq→UniProt position map.** AlphaMissense is keyed by the UniProt canonical sequence, but this project uses RefSeq numbering. 15/16 genes align perfectly, but **CHRNA1**'s UniProt entry (P02708, 482 aa) carries a 25-aa P3A alternative-splice exon the RefSeq isoform (NP_000070.1, 457 aa) lacks. A `difflib` alignment builds a per-gene RefSeq→UniProt map (identity for 15 genes, +25 offset after ~pos 78 for CHRNA1), applied before lookup.
2. **Wildtype cross-check.** A variant whose recorded wildtype doesn't match AlphaMissense's wildtype at the (mapped) position is a data-entry error; its score is imputed (in-batch mean) rather than force-mapped.

Final: **687/717 (95.8%) clean matches**, 0 missing; 30 rows (4.2%) are genuine data-entry errors, neutral-imputed.

## Key takeaways

**AlphaMissense is a null on both framings — mean Δ +0.003 (3-class) and +0.006 (binary), both inside fold noise.** This is exactly what Run VIII predicted: AlphaMissense measures *pathogenicity* (damaging vs benign), which is **orthogonal to direction of effect** (GOF vs LOF). A "damaging GOF" and a "damaging LOF" variant get near-identical AlphaMissense scores, so it can't separate them.

**The 1-seed ablation agrees.** Dropping `alphamissense` *improves* RF macro-F1 in both framings (+0.018 3-class, +0.021 binary) — i.e. it is pure noise for this task. The only features that consistently help when dropped are the base physicochemical / substitution / positional groups (negative delta).

**Verdict: remove AlphaMissense.** It earns no F1 and, per the "don't keep a feature that doesn't improve the score" rule, should be dropped from the default pipeline. This is itself a *positive thesis finding*: direction-of-effect is a distinct problem from pathogenicity, and stacking a SOTA pathogenicity score on top of the direction model adds nothing.

**Caveat on the ablation.** It runs **1 seed** (seed 42) for speed, so its per-group deltas are point estimates (±0.01–0.03 noise); the 3-seed compare above is the controlled number for AlphaMissense, and it says null. The same 1-seed pass shows ESM-2 and conservation as slightly *negative* — at odds with Run VIII's 3-seed +0.023 for the pair — so treat those two as "needs a 3-seed re-ablation", not settled.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ============ 3-class: Run VIII vs Run X (gene-grouped subunit CV) ============
# 71 feats = results/comparison/comparison_subunit_features_v2.csv    (Run VIII)
# 72 feats = results/comparison/comparison_subunit_alphamissense.csv  (Run X)
run8 = {"Logistic Regression": 0.4834, "SVM (RBF)": 0.4099, "Random Forest": 0.5012,
        "LightGBM": 0.4750, "SVM (Linear)": 0.4745, "KNN": 0.4651,
        "Gaussian NB": 0.4586, "MLP": 0.4669, "XGBoost": 0.4695, "CatBoost": 0.5081}
runX = {"Logistic Regression": 0.4995, "SVM (RBF)": 0.4462, "Random Forest": 0.5028,
        "LightGBM": 0.4800, "SVM (Linear)": 0.4705, "KNN": 0.4657,
        "Gaussian NB": 0.4657, "MLP": 0.4475, "XGBoost": 0.4717, "CatBoost": 0.4889}
d3 = pd.DataFrame([{"Model": m, "Run VIII (71)": run8[m], "Run X (72)": runX[m],
                    "Δ F1": runX[m] - run8[m]} for m in run8]
                 ).sort_values("Run X (72)", ascending=False).reset_index(drop=True)
d3["Δ F1"] = d3["Δ F1"].map(lambda x: f"{x:+.3f}")
d3.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("3-class mean F1 — Run VIII: %.4f | Run X: %.4f | Δ: %+.4f\n" % (
    np.mean(list(run8.values())), np.mean(list(runX.values())),
    np.mean(list(runX.values())) - np.mean(list(run8.values()))))
display(d3)

# ============ Binary GOF/LOF: Run VIII vs Run X ============
run8b = {"Logistic Regression": 0.5978, "SVM (RBF)": 0.5675, "Random Forest": 0.6469,
         "LightGBM": 0.6169, "SVM (Linear)": 0.5793, "KNN": 0.5858,
         "Gaussian NB": 0.5853, "MLP": 0.5955, "XGBoost": 0.6245, "CatBoost": 0.6366}
runXb = {"Logistic Regression": 0.6156, "SVM (RBF)": 0.5639, "Random Forest": 0.6304,
         "LightGBM": 0.6263, "SVM (Linear)": 0.6030, "KNN": 0.5970,
         "Gaussian NB": 0.5925, "MLP": 0.6067, "XGBoost": 0.6447, "CatBoost": 0.6121}
db = pd.DataFrame([{"Model": m, "Run VIII (71)": run8b[m], "Run X (72)": runXb[m],
                    "Δ F1": runXb[m] - run8b[m]} for m in run8b]
                 ).sort_values("Run X (72)", ascending=False).reset_index(drop=True)
db["Δ F1"] = db["Δ F1"].map(lambda x: f"{x:+.3f}")
db.index = ["🥇", "🥈", "🥉", "4", "5", "6", "7", "8", "9", "10"]
print("Binary mean F1 — Run VIII: %.4f | Run X: %.4f | Δ: %+.4f\n" % (
    np.mean(list(run8b.values())), np.mean(list(runXb.values())),
    np.mean(list(runXb.values())) - np.mean(list(run8b.values()))))
display(db)

# ============ Leave-one-group-out ablation (Random Forest, 1 seed) ============
# negative delta = feature HELPS (dropping it lowers F1); positive = HURTS.
# 1 seed only -> point estimate (noise ~ ±0.01-0.03); the 3-seed compare above is the controlled number.
abl3 = pd.DataFrame({
    "Condition": ["drop_positional", "drop_physicochemical", "drop_substitution",
                  "drop_conservation", "drop_structural_nachr", "drop_esm2",
                  "drop_conformational", "drop_alphamissense", "drop_structural_core"],
    "3-class ΔF1": [-0.063, -0.013, -0.006, -0.005, +0.012, +0.013, +0.014, +0.018, +0.034],
    "binary ΔF1":  [+0.010, -0.019, -0.057, +0.015, +0.015, +0.026, +0.015, +0.021, -0.053],
}).set_index("Condition").sort_values("3-class ΔF1")
print("Full-model (1-seed) F1 — 3-class 0.4494 | binary 0.6234")
display(abl3)

---
## Run XI: 4 mechanism features (82 features) — neutral-to-negative; gene identity is the only signal

**Date:** 2026-08-18
**Config:** 717 variants, **82 features**, subunit CV (gene-grouped, no leakage), Random Forest, 1 seed (42), 5 outer folds, 30 Optuna trials.
**Change vs Run X:** AlphaMissense dropped (Run X null). Four mechanism/structure feature groups added:

| Group | n | What it encodes |
|-------|---|-----------------|
| `stability` | 2 | burial-weighted ΔΔG proxy — `(hyd_wt−hyd_mt)·burial`, `|vol_mt−vol_wt|·burial` (the ESM-IF/ThermoMPNN slot, without the broken torch_geometric dep) |
| `functional_motif` | 5 | flags: in TM / in M2 / in gate ring / in Cys-loop / in agonist-binding aromatic box |
| `charge_context` | 4 | signed charge change × {whole, TM, M2, binding-site} |
| `conformational` | 5 | real α7 open↔closed deltas (RSA, B-factor, interface, pore radius, CA RMSD) — 4/5 were stubbed to 0 |

Total 82 features across 11 groups (was 72 across 10).

> **Why these four.** Run VIII proved the effect-vs-no-effect axis (conservation, ESM-2) lifts 3-class but is blind to GOF↔LOF *direction*. These four target the direction gap directly: where the mutation lands in the receptor (motif, charge-context), what it does to folding (stability), and whether it biases the open/closed equilibrium (conformational). They are the nAChR-mechanism hypothesis made explicit.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# Run XI: leave-one-group-out ablation, Random Forest, 1 seed (42), 5 outer folds, 30 trials.
# Full-model (82 feats) macro F1 = 0.4487. delta_f1 = F1(drop) - F1(full); NEGATIVE = helps.
# 1 seed only -> point estimate (macro_f1_std = 0.0). Read small deltas as noise, not ranking.
ablation_xi = pd.DataFrame([
    {"Condition": "drop_positional",        "ΔF1 (3-class)": -0.0578, "read": "🔴 critical — only real signal"},
    {"Condition": "drop_conservation",      "ΔF1 (3-class)": -0.0001, "read": "⚪ neutral"},
    {"Condition": "drop_structural_core",   "ΔF1 (3-class)": +0.0002, "read": "⚪ neutral"},
    {"Condition": "drop_functional_motif",  "ΔF1 (3-class)": +0.0038, "read": "🟢 NEW — neutral"},
    {"Condition": "drop_physicochemical",   "ΔF1 (3-class)": +0.0090, "read": "🟢 (1-seed noise; Run II: −0.016)"},
    {"Condition": "drop_charge_context",    "ΔF1 (3-class)": +0.0108, "read": "🟢 NEW — neutral"},
    {"Condition": "drop_structural_nachr",  "ΔF1 (3-class)": +0.0139, "read": "🟢 neutral"},
    {"Condition": "drop_conformational",    "ΔF1 (3-class)": +0.0142, "read": "🟢 NEW — neutral"},
    {"Condition": "drop_esm2",              "ΔF1 (3-class)": +0.0143, "read": "🟢 (1-seed noise; Run VIII: helps)"},
    {"Condition": "drop_stability",         "ΔF1 (3-class)": +0.0201, "read": "🟢 NEW — neutral"},
    {"Condition": "drop_substitution",      "ΔF1 (3-class)": +0.0239, "read": "🟢 (1-seed noise; Run II: −0.017)"},
]).set_index("Condition")
ablation_xi["ΔF1 (3-class)"] = ablation_xi["ΔF1 (3-class)"].map(lambda x: f"{x:+.4f}")
display(ablation_xi)

### Run XI (3-class ablation) Key Takeaways

**The four new features are all neutral-to-negative on the honest 1-seed RF metric.** Full-model F1 = 0.4487 (≈ Run X's 0.4494 at 1 seed — swapping AlphaMissense out for 11 mechanism features is a wash). Dropping any one of the four *improves* F1 by +0.004…+0.020, i.e. each is, at best, dead weight and, at worst, mild noise.

1. **The only robust signal is gene identity.** `drop_positional` = −0.058 — the largest move by 2.5×, matching every run since Run I. Which subunit a variant sits in is what the model actually learns.

2. **⚠️ Read the small deltas as noise, not ranking.** In this single 1-seed pass, *substitution* (+0.024) and *physicochemical* (+0.009) also show up as "harmful" — directly contradicting Run II's 10-model 3-seed result (−0.017 / −0.016, the #2/#3 features). A 1-seed RF point estimate cannot rank features inside a ±0.02 band. So the honest claim is not "stability is worse than functional_motif" — it's "all four sit at or below zero; none helps."

3. **The mechanism hypothesis did not land.** Run VIII's framing was that direction needs nAChR-mechanism signal (open/closed conformation, ligand/interface context). Four concrete attempts at that signal — motif flags, charge-in-context, burial-weighted stability, real α7 conformational deltas — none moved GOF↔LOF. Hand-crafted receptor-geometry features are too coarse to separate gain- from loss-of-function; the direction signal is not in *where* the residue sits or *how buried* it is.

4. **Remaining levers are unchanged and now the only ones left.** Data volume (cross-species augmentation — the next step) and learned (not hand-crafted) representations. The feature-engineering well is, at this point, dry.

### Verdict

Drop all four new groups from the default pipeline (per the "don't keep a feature that doesn't improve the score" rule). If a single one is worth keeping for thesis narrative, it's `conformational` — the real α7 deltas are scientifically defensible and were already in the feature list (just stubbed) — but it earns no F1 either.

*(Binary GOF/LOF ablation skipped at the user's request.)*

---
## Run XII: fix `structural_nachr` (5 dead features) — full F1 +0.010 (1-seed), `positional` still the only robust signal

**Date:** 2026-08-26 · **Model:** Random Forest · **CV:** nested leave-one-subunit-out (gene-grouped, no leakage), 30 Optuna trials · **Data:** 717 variants, 71 features.

**Change vs Run XI:** reverted to the 71-feature default (Run XI's four mechanism groups dropped), then **fixed two extractors** — no new features added.

1. `structural_nachr` computed 5 of its 7 features wrong (raw UniProt position used as a PDB residue key → pore/interface/burial silently 0.0; binding-site coords never populated; `subunit_burial` was a `return 0.0` placeholder; TM table only 10/16 genes). Rebuilt on `mechanism.py` + the shared alignment maps.

| feature | before | after |
|---|---|---|
| `pore_distance` | ~0 (wrong key) | **91.5%** non-zero |
| `ligand_proximity` | 0 (never populated) | **48.4%** non-zero |
| `interface_proximity` | ~0 (wrong key) | **88.1%** non-zero |
| `interface_contacts` | 0.8% (CA-only cutoff) | **31.4%** non-zero (heavy-atom) |
| `subunit_burial` | constant 0 (placeholder) | **31.4%** non-zero |
| `tm_helix` / `tm_depth` | 10/16 genes | **16/16** genes |

2. `conformational` had `ca_rmsd` ↔ `pore_radius_change` swapped (columns 3↔4). Value-invariant under Random Forest (column order doesn't change a tree ensemble) → **no performance effect**, interpretability only.

In [ ]:
import pandas as pd
from IPython.display import display

# Run XII — OVERALL full-model performance (not per-group deltas).
# pre-fix  = ablation_3class_71feat.log (seed 42, old broken structural_nachr)
# 1-seed   = post-fix, seed 42
# 5-seed   = post-fix, seeds {42, 123, 456, 789, 1011}, mean ± std across seeds
#
# Macro F1 is the headline metric: classes are imbalanced (LOF 363 / NNE 164 / GOF 190),
# so plain accuracy rewards predicting the LOF majority.

overall = pd.DataFrame({
    "metric": ["Macro F1 (headline)", "MCC", "Balanced accuracy", "Accuracy",
               "Macro precision", "Macro recall"],
    "pre-fix (1-seed)":  [0.4569, 0.2552, 0.5104, 0.5199, 0.4919, 0.4786],
    "post-fix (1-seed)": [0.4665, 0.2752, 0.5346, 0.5292, 0.5112, 0.4975],
    "post-fix (5-seed)": ["0.4931 ± 0.025", "0.3032 ± 0.036", "0.5214 ± 0.024",
                          "0.5628 ± 0.026", "0.5195 ± 0.028", "0.5140 ± 0.024"],
}).set_index("metric")

display(overall)
print("5-seed is the trustworthy level — seed 42 alone is a low seed.")

**Reading the overall numbers.** The fix lifts macro F1 **+0.010** (0.4569 → 0.4665) at 1 seed, and the multi-seed level settles at **0.4931 ± 0.025** — the 5-seed mean is the number to quote, since seed 42's point estimate (0.4665) understates the model. MCC (+0.020) and balanced accuracy (+0.024) move together with F1, so the gain is a real signal, not an accuracy artifact.

Context for the headline metric: with a 363 / 164 / 190 class split, a majority-class (LOF) baseline scores ~0.51 accuracy but only **0.33 balanced accuracy** (one class predicted). The model's 0.52 balanced accuracy is above that chance floor, but the gap to a clinically useful predictor is still wide — that's the whole motivation for adding data, not features.

In [ ]:
import pandas as pd
from IPython.display import display

# Run XII — leave-one-group-out ablation (Random Forest, subunit CV, 30 trials).
# Transcribed from results/ablation/ablation_impact_random_forest.csv.
# delta_f1 = F1(drop group) - F1(full); NEGATIVE = group helps (dropping it lowers F1).

r12 = pd.DataFrame({
    "group": ["positional", "conservation", "structural_nachr", "substitution",
              "physicochemical", "structural_core", "conformational", "esm2"],
    "dF1_1seed": [-0.0557, -0.0131, -0.0118, -0.0059, -0.0012, +0.0034, +0.0120, -0.0066],
    "dF1_5seed": [-0.0124, -0.0033, -0.0019, +0.0009, +0.0066, +0.0070, +0.0071, +0.0096],
    "std_5seed": [0.041, 0.024, 0.026, 0.030, 0.029, 0.017, 0.019, 0.030],
}).set_index("group").sort_values("dF1_5seed")

r12_fmt = r12.copy()
for c in ["dF1_1seed", "dF1_5seed", "std_5seed"]:
    r12_fmt[c] = r12_fmt[c].map(lambda x: f"{x:+.4f}")
display(r12_fmt)

print("\nFull-model macro F1: 1-seed 0.4665 (pre-fix 0.4569, +0.010) | 5-seed 0.4931 ± 0.025")
print("drop_conformational -> 0.5002 (vs full 0.4931): dropping it gives ~+0.007")

### Run XII takeaways

1. **The fix did its job, but it's a cleanup, not a new lever.** `structural_nachr` flipped from *harmful* in Run XI (+0.014, dead columns) to neutral/slightly-helpful here (−0.002 ± 0.026). The five zero-columns now carry signal, so the group no longer drags the model. Full F1: **0.4569 → 0.4665** (1-seed) and **0.4931 ± 0.025** (5-seed).

2. **`positional` (gene identity) is the only robust signal — again.** −0.012 is the largest delta and the only one that clears its own std. This has held across every run: the model is effectively "which subunit is it" plus weak residue-level signal.

3. **Conformational: DROP.** `drop_conformational` = +0.007 (removing it *helps*) at 5 seeds, +0.012 at 1 seed, and it is CHRNA7-only (5 columns = 0.0 for ~93% of variants). Removed from the default pipeline → **66 features / 7 groups**. Kept as an importable module for the thesis ablation section.

4. **The 1-seed ↔ 5-seed flip is the point of the confirmation.** `substitution`, `physicochemical`, `structural_core`, `esm2` all change sign between 1 and 5 seeds — that's the ±0.02–0.03 noise band, not feature ranking. Only `positional` (reliably negative) and `conformational` (structural case for dropping) are settled; the rest need data, not more features.

---
## Run XIII: "new ideas" screen — embeddings, stacking, ordinal, LoRA all null or negative

**Date:** 2026-09-02 · **CV:** nested leave-one-subunit-out (gene-grouped, no leakage), 30 Optuna trials, 1 seed (42) · **Data:** 760 multi-species variants (human 507 / rat 173 / mouse 80; LOF 377 / NNE 188 / GOF 195), 66 baseline vs 82 full features.

**Purpose:** after Run XII declared the hand-crafted feature well dry, this screen tests the four *learned / architectural* ideas in one pass. Every idea is compared against the 66-feature baseline on identical data and seed, so the deltas below are clean A/Bs.

| # | Idea | Result (macro F1) | Verdict |
|---|---|---|---|
| #1 | LoRA fine-tune ESM-2 150M (r=8) | **0.267** | below the 0.33 chance floor — strong negative |
| #2 | Stability ΔΔG via RaSP | — | BLOCKED (not pip-installable) |
| #3 | Per-residue ESM-2 embeddings (+16 PCA) | Δ ≈ +0.001 | null |
| #4 | Ensemble stacking (logistic meta-learner) | 0.463 | below catboost 0.490 — null |
| #5 | Ordinal regression (cumulative-link) | 0.378 baseline | worst of six — firm negative |


In [ ]:
import pandas as pd
from IPython.display import display

# Run XIII — 1-seed 3-class screen. Reads the per-model CSV written by run_ideas.py
# (results/new_ideas/ideas_1seed.csv). baseline = 66 features; +embeddings = 82
# (adds 16 per-residue ESM-2 PCA components). Macro F1 is the headline metric.

r13 = pd.read_csv("results/new_ideas/ideas_1seed.csv")

pivot = r13.pivot_table(index="model", columns="condition",
                        values="macro_f1_mean", aggfunc="first")
pivot["delta_f1"] = pivot["+embeddings"] - pivot["baseline"]
pivot = pivot[["baseline", "+embeddings", "delta_f1"]].sort_values("baseline", ascending=False)

pivot_fmt = pivot.copy()
for c in ["baseline", "+embeddings", "delta_f1"]:
    pivot_fmt[c] = pivot_fmt[c].map(lambda x: f"{x:+.4f}" if pd.notna(x) else "—")
display(pivot_fmt)

print("\nBest baseline: catboost 0.4900.")
print("Stacking (full features only) = 0.4629 — below the best single model.")
print("LoRA (separate script, lora_results.csv) = 0.2668 — below the 3-class chance floor.")


**Reading the numbers.** Each idea is an *addition* to an already-strong baseline (catboost 0.490, in the same band as Run XII's 0.493 on 717 human variants — different data, so treat as a level, not an exact compare). None clears it:

- **Embeddings (#3) — a wash.** Mean Δ +0.001, but the sign is model-class-dependent: the linear models gain (ordinal +0.034, LR +0.012) while the tree models that actually win lose (lightgbm −0.038, catboost −0.015). The 16 PCA components are noise the trees overfit, and they only lift models too weak to matter.
- **Ordinal (#5) — firm negative.** 0.378 baseline is the worst of all six by a wide margin; the `LOF < NNE < GOF` ordering assumption is actively wrong. NNE (188 variants) sits ambiguously between the two functional extremes, so forcing a monotone link discards signal.
- **Stacking (#4) — does not beat the best single model.** 0.463 < catboost 0.490; the meta-learner averages the weaker models down instead of extracting a complement.
- **LoRA (#1) — the clearest negative.** 0.267 is below the 0.33 chance floor: fine-tuning a 150M-param ESM-2 on 760 variants (grouped by gene) is too little data, and it overfits worse than classical ML. Strongest evidence yet that learned representations cannot rescue a dataset this small.


### Run XIII takeaways

1. **Every remaining non-data idea is ruled out.** Embeddings (null), stacking (null), ordinal (negative), LoRA (strongly negative), RaSP (blocked). Combined with Run XII's four neutral-to-negative mechanism groups, the feature and architecture space is exhausted.

2. **The ceiling is the data, not the model.** catboost 0.490 (1-seed, 760 multi-species variants) is the honest number, and no lever that avoids new data moved it.

3. **Learned representations are orthogonal to direction, at every scale.** Zero-shot ESM-2 scores (Run VIII), per-residue embeddings (here), and full LoRA fine-tuning (here) all fail to separate GOF↔LOF — reaffirming that direction-of-effect is a distinct problem from pathogenicity, and that 760 labels is far below what even parameter-efficient transfer needs.

4. **The next step is more curated data, not more models.** The 760 variants already span human/rat/mouse, so the open lever is expanding the curated variant count (datascraper/ + human-automation pipelines), not fancier features or architectures.


## Run XIV: fix structural position→PDB mapping + drop `structural_nachr`

The "structural features don't help" question had a concrete answer: the position→PDB
mapping was broken, so `structural_core` was reading features from the *wrong residues*.
Fixing it removes the harm but — consistent with Run XIII — does not turn structure into
a strong direction signal. `structural_nachr` is dropped from the default set (sparse, neutral).


### What was broken

1. **5 genes homology-mapped to a different subunit** — CHRNA2/5/6, CHRNB3, CHRNG were mapped to
   another gene's chain (~47% identity), so every structural feature came from the wrong residue.
2. **CHRNA10 used the wrong isoform** — AF-Q13002 (908 aa) instead of the canonical Q9GZZ6 (450 aa).
3. **The 7 AlphaFold models had no `.dssp`** — RSA fell back to a fill value (1.0) for every position.

### Fix

- Gapped global alignment (BioPython `PairwiseAligner`, BLOSUM62, open −10 / extend −0.5) +
  **<80% identity rejection** in `structural.py::_build_alignment_map`.
- Repointed 6 genes to their **own AlphaFold model** in `config.py::PDB_MAPPING`.
- Real solvent accessibility via **Shrake-Rupley SASA** in `structural.py::_compute_dssp_dict_from_structure`
  (bypasses `mkdssp`, not installed).

Result: all 760 variants map to real RSA (91.6% overall; the previously-broken genes 100%).


In [ ]:
import pandas as pd
from IPython.display import display

# Run XIV — structural_core mapping fix + structural_nachr removed from default.
# Reads results/ablation/ablation_impact_random_forest.csv (75 features, RF, 1 seed,
# gene-grouped subunit CV, 30 trials). delta_f1 = F1(drop) - F1(full); NEGATIVE = helps.
# 1 seed -> point estimate (macro_f1_std = 0.0); read deltas as ~±0.02 noise, not a ranking.

r14 = pd.read_csv("results/ablation/ablation_impact_random_forest.csv", index_col=0)
r14 = r14[["n_features", "macro_f1_mean", "delta_f1"]].sort_values("delta_f1")
r14_fmt = r14.copy()
r14_fmt["macro_f1_mean"] = r14_fmt["macro_f1_mean"].map(lambda x: f"{x:.4f}")
r14_fmt["delta_f1"] = r14_fmt["delta_f1"].map(lambda x: f"{x:+.4f}")
display(r14_fmt)

# structural_core before/after the mapping fix (broken -> fixed), RF 1-seed:
#   broken mapping (Run XII, 71-feat):  +0.007  (structural hurt)
#   fixed mapping  (Run XIV, 75-feat):  -0.002  (neutral)
print("\nstructural_core delta:  +0.007 (broken) -> -0.002 (fixed).")
print("Full-model macro F1: 0.4597 (82-feat, with structural_nachr) -> 0.4715 (75-feat, nachr removed).")


### Run XIV takeaways

1. **The mapping bug is fixed; structure is now *neutral*, not harmful.** `structural_core` flipped
   from +0.007 (broken mapping → wrong-residue features) to −0.002 (fixed → correct residues +
   real RSA). Fixing the bug removed the drag but did not create a positive signal — structure is
   still not a direction lever, consistent with Run XIII.

2. **The earlier 82-feat run overstated the structural win.** In that run `structural_core` showed
   −0.019, but that was redundancy with the (harmful) `structural_nachr` group. Once `structural_nachr`
   is dropped, `structural_core` reverts to −0.002. Read per-group deltas with the *full* feature set in mind.

3. **`structural_nachr` removed from the default set.** Its pore/interface/burial features are 0.0 for the
   7 AlphaFold-monomer genes (single chain → no interface) and its ligand feature is α-only
   (~72% of the group is zeros). Dropping it lifts full F1 0.4597 → 0.4715 and simplifies 82 → 75
   features. Module kept for the thesis ablation section.

4. **The ceiling is unchanged: data.** This is a correctness cleanup, not a new lever — the headline
   number is still catboost 0.490 (Run XIII). The structural feature space is now clean (correct mapping,
   real ASA, no degenerate group); the open lever remains curated-data expansion.


## Run XV: structural overhaul (mutation-aware features) — NEGATIVE, reverted

Run XIV fixed the position→PDB mapping so `structural_core` reads the *right* residue,
but it stayed neutral (−0.002). Run XV attacks the question head-on: diagnose every reason
structure is weak, fix each one, and re-measure.


### The 5 reasons structural features were weak, and the fixes

1. **WT-only features.** `structural_core` (RSA, Cβ-density, B-factor, DSSP, is_unmappable)
   describes the *wild-type* residue, so every substitution at a position gets an identical
   value → no mutation signal. **Fix:** add mutation-aware extractors.
2. **B-factor scale mixing.** pLDDT (AlphaFold) vs crystallographic B (experimental PDBs) are
   different scales, blended into one column. **Fix:** z-score within each structure type.
3. **`is_unmappable` is ~92% dead.** Only ~8% of variants are unmappable → a near-constant column.
4. **Missing nAChR biology.** No M2-gate / Cys-loop / binding-pocket / TM membership.
   **Fix:** FunctionalMotifExtractor — 5 binary flags at the GOF/LOF functional sites.
5. **No burial × chemical-change interaction.** A buried Ala→Val matters more than a surface
   one. **Fix:** StabilityExtractor — burial × Δhydrophobicity and burial × Δvolume (a
   structure-aware ΔΔG proxy).

**Three extractors wired in:** StabilityExtractor (2), FunctionalMotifExtractor (5),
ChargeContextExtractor (4 — Δcharge × location). Plus structural_core expanded to 9 features
(pLDDT + contact_number + z-scored B-factor) → 88-feat "full overhaul".


In [ ]:
import pandas as pd
from IPython.display import display

# Run XV — structural overhaul. Two configs measured (1-seed, macro F1, gene-grouped CV):
#   88-feat "full overhaul"  (structural_core→9 + stability + functional_motif + charge_context)
#   82-feat "refined"        (structural_core→7 reverted + charge_context dropped)
# The 82-feat run was the last on-disk state; preserved as *_run15.csv before reverting to 75-feat.
# delta_f1 = F1(drop) - F1(full); NEGATIVE = group helps. 1 seed -> ±0.02 noise, not a ranking.

r15_rf = pd.read_csv("results/ablation/ablation_impact_random_forest_run15.csv", index_col=0)
r15_cb = pd.read_csv("results/ablation/ablation_impact_catboost_run15.csv", index_col=0)

def fmt(df):
    d = df[["n_features", "macro_f1_mean", "delta_f1"]].sort_values("delta_f1").copy()
    d["macro_f1_mean"] = d["macro_f1_mean"].map(lambda x: f"{x:.4f}")
    d["delta_f1"] = d["delta_f1"].map(lambda x: f"{x:+.4f}")
    return d

print("82-feat RF  (full F1 %.4f):" % r15_rf.loc["full", "macro_f1_mean"])
display(fmt(r15_rf))
print("\n82-feat CatBoost  (full F1 %.4f):" % r15_cb.loc["full", "macro_f1_mean"])
display(fmt(r15_cb))

# Headline comparison (1-seed macro F1):
print("\n  config        |   RF    | CatBoost")
print("  75-feat base  | 0.4715  | 0.4773   <- best, kept")
print("  88-feat full  | 0.4581  | 0.4672")
print("  82-feat ref   | 0.4577  | 0.4468")


### Run XV takeaways

1. **The structural overhaul did not flip it — it regressed it.** Every mutation-aware /
   structure-expanded config scored *below* the 75-feat baseline on both models. The
   "helpful" ablation deltas for stability/functional_motif were ~±0.02 noise: they flipped
   sign between the 88-feat and 82-feat runs (1 seed → macro_f1_std = 0.0).

2. **Conclusion: single-residue structural context carries no direction-of-effect signal for
   nAChR GOF/LOF — WT-side or mutation-aware.** The 5 diagnosed reasons were real and fixed,
   yet none of the fixes produced a robust lift. The signal is *absent*, not *broken*.

3. **Repo reverted to the 75-feat default set** (Run XIV baseline, catboost 0.490). The three
   mutation-aware modules (stability.py, functional_motif.py, charge_context.py) are kept on
   disk for the thesis ablation section but are NOT in the default set.

4. **Where the direction signal actually lives** (consistent with Run XIII): sequence-level
   conservation + ESM-2 embeddings + positional one-hot. Structure is not a lever; the
   remaining headroom is the data (curated variants + label quality), not features.


## Run XVI: α7-only structural diagnostic — the professor's two-state hypothesis, tested

Runs XII–XV settled that single-residue structural context is neutral/negative in the **full
16-gene model** — where most genes lack a dedicated structure and the receptor is pentameric.
The professor's counter-proposal: *run CHRNA7 (α7) alone* — it's **homomeric** (no interface
ambiguity), the most-represented gene, and the **only one with both a closed (7EKT) and open
(7KOX) experimental structure** — and if the two-state signal works there, later "map other
genes to α7 by approximation."

This run tests that hypothesis directly, in the one gene where complete two-state structure
has the best possible chance to carry signal.


### Setup

- **Data:** `load_mutation_data(subunits=["CHRNA7"])` → 200 variants (human 119 / rat 80 /
  mouse 1). Labels: LOF 100 / No-net-effect 58 / GOF 42.
- **Features:** 75-feat default **+ ConformationalExtractor** (the α7-unique open↔closed
  delta group, NOT in the default set) = 80 features. The 16 dead `subunit_*` one-hots are
  masked (constant for a single gene) → 64 features.
- **CV:** `nested_cross_validation(cv_mode="standard")` — plain StratifiedKFold (gene grouping
  is meaningless for one gene), **3 seeds** so `macro_f1_std` is a real confidence interval,
  not the 1-seed `std=0` that plagued Runs XIV/XV.
- **Confound control:** the `species_*` columns carry a human↔rat↔label shortcut (human-α7 is
  84 LOF / 16 GOF; rat-α7 is 39 NNE / 25 GOF), so a `no_species` run masks them out.


In [ ]:
import pandas as pd
from IPython.display import display

# Run XVI — α7-only, 3-class, all-species (200 variants), 3 seeds.
# delta_f1 = F1(drop) - F1(full); NEGATIVE = the dropped group helps.

r16_cb = pd.read_csv("results/alpha7/alpha7_diagnostic_catboost.csv", index_col=0)
r16_rf = pd.read_csv("results/alpha7/alpha7_diagnostic_random_forest.csv", index_col=0)

def fmt(df):
    d = df[["n_features", "macro_f1_mean", "macro_f1_std", "delta_f1"]].sort_values("delta_f1").copy()
    d["macro_f1_mean"] = d["macro_f1_mean"].map(lambda x: f"{x:.4f}")
    d["macro_f1_std"]  = d["macro_f1_std"].map(lambda x: f"{x:.4f}")
    d["delta_f1"]      = d["delta_f1"].map(lambda x: f"{x:+.4f}")
    return d

print("CatBoost α7-only  (full F1 %.4f ± %.4f, 3 seeds):" % (
    r16_cb.loc["full", "macro_f1_mean"], r16_cb.loc["full", "macro_f1_std"]))
display(fmt(r16_cb))
print("\nRandomForest α7-only  (full F1 %.4f ± %.4f, 3 seeds):" % (
    r16_rf.loc["full", "macro_f1_mean"], r16_rf.loc["full", "macro_f1_std"]))
display(fmt(r16_rf))

# The crux, both models side by side:
print("\n  open<->closed conformational delta (drop_conformational - full):")
print("    CatBoost     %+.4f  (± %.4f)" % (
    r16_cb.loc["drop_conformational", "delta_f1"], r16_cb.loc["drop_conformational", "macro_f1_std"]))
print("    RandomForest %+.4f  (± %.4f)" % (
    r16_rf.loc["drop_conformational", "delta_f1"], r16_rf.loc["drop_conformational", "macro_f1_std"]))


### Run XVI takeaways — the hypothesis is settled

1. **α7 alone "works" at the surface level: macro F1 ≈ 0.60 / 0.58** (catboost / RF). But this
   is *not* evidence for structure — it's the single-gene + species shortcut inflating the score
   (a 3-class task on 200 variants with a human-84-LOF / rat-39-NNE split is an easier, partially
   species-leakable problem than the full 16-gene task).

2. **The open↔closed conformational group is null even in α7.** `drop_conformational` = −0.004
   (catboost, ±0.041) and −0.013 (RF, ±0.029). The std is 3–10× the effect in both models. In the
   *one* homomeric gene with *complete* two-state experimental structure (7EKT closed → 7KOX open),
   the conformational delta does not predict GOF/LOF/NNE direction. This is the direct, clean test
   of the professor's hypothesis — and it fails.

3. **structural_core is weak and sign-flips** (−0.015 catboost vs +0.011 RF) — the same noise we
   saw in Runs XIII–XV, now confirmed with real seed variance in the most structure-rich gene.

4. **The species confound is real but model-dependent** (`no_species` = −0.022 catboost vs
   +0.003 RF): catboost exploits the human↔rat↔label shortcut more than RF does. Either way it
   does not dominate — physicochemical (−0.030) is the strongest lever, and that is *sequence*
   chemistry, not structure.

**Bottom line for the professor:** the direction signal is genuinely *absent* from single-residue
structure, even in the ideal homomeric two-state case. The "map other genes to α7 by approximation"
path is therefore moot — mapping to a structure that carries no signal cannot create one. The lever
is data (more curated variants, label quality), not structure. The conformational/structural modules
stay on disk for the thesis ablation section but remain out of the default set.
